In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from datasets import load_dataset
import pandas as pd

# Laden des Datasets
dataset = load_dataset("MuskumPillerum/General-Knowledge")


2024-12-14 15:50:01.617920: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734187801.629460  286676 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734187801.632775  286676 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-14 15:50:01.644742: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/tobias/anaconda3/envs/taprojekt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please 

In [7]:
df = pd.DataFrame(dataset['train'])
df.isna().value_counts()

Answer  Question
False   False       37623
True    False          12
Name: count, dtype: int64

In [9]:
df = df.dropna()
df.isna().value_counts()

Answer  Question
False   False       37623
Name: count, dtype: int64

In [10]:

# Beispielhafte Vorbereitung der Daten (du müsstest dies für alle Daten tun)
texts = [example['Question'] for example in df.loc[:100]]  # nur die ersten 100 Fragen für dieses Beispiel
labels = [example['Answer'] for example in df.loc[:100]]


TypeError: string indices must be integers, not 'str'

In [ ]:

# Tokenisierung
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts + labels)  # Tokenizer für sowohl Fragen als auch Antworten
question_sequences = tokenizer.texts_to_sequences(texts)
answer_sequences = tokenizer.texts_to_sequences(labels)

vocab_size = len(tokenizer.word_index) + 1

# Padding
max_question_length = max(len(seq) for seq in question_sequences)
max_answer_length = max(len(seq) for seq in answer_sequences)

padded_questions = pad_sequences(question_sequences, maxlen=max_question_length, padding='post')
padded_answers = pad_sequences(answer_sequences, maxlen=max_answer_length, padding='post')

# Modellarchitektur
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, input_length=max_question_length),
    tf.keras.layers.LSTM(64, return_sequences=False),
    tf.keras.layers.Dense(max_answer_length * vocab_size, activation='softmax'),
    tf.keras.layers.Reshape((max_answer_length, vocab_size))
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

# Anpassung der Labels für das Training
# Hier ist ein vereinfachtes Beispiel; in der Praxis müsstest du den Output in eine Sequenz von Wahrscheinlichkeiten für jedes Wort in der Antwort umwandeln
model.fit(padded_questions, padded_answers, epochs=10, verbose=1)